In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import math
import pysam
from sklearn.model_selection import train_test_split

In [2]:
genome = pysam.FastaFile('../../results/marand_2021_scATAC/AGPv4.fa')

In [3]:
for window_size in [300, 600, 1000]:
    atac_df = pd.read_csv('../../results/marand_2021_scATAC/GSE155178_maize_scATAC_atlas_ACR_celltype_CPM.txt', 
                          header=0, sep='\t').reset_index().rename(columns={'index': 'locus'})
    
    locus_parts = atac_df['locus'].str.split('_', expand=True).astype({1: int, 2: int})
    locus_parts.columns = ['seqid', 'start', 'end']
    
    midpoints = ((locus_parts['start'] + locus_parts['end']) // 2)
    locus_parts['start'] = midpoints - (window_size // 2)
    locus_parts['end'] = midpoints + (window_size // 2)
    
    atac_data = pd.concat([locus_parts, atac_df.drop(columns='locus')], axis='columns')
    atac_data = atac_data.sort_values(by=['seqid', 'start']).set_index(['seqid', 'start', 'end'])
    
    atac_binary = (atac_data > math.log2(5)).astype(int)
    
    unlabeled_bed = pd.read_csv(
        f'../../results/marand_2021_scATAC/c{window_size}/scATAC.unlabeled.bed', 
        header=None, 
        sep='\t',
        names=['seqid', 'start', 'end', 'label']
    )
    
    # Merge datasets and prepare for processing
    merged_data = pd.merge(unlabeled_bed, atac_binary, on=['seqid', 'start', 'end'], how='left')
    merged_data = merged_data.fillna(0)
    merged_data = merged_data.drop(columns='label')
    
    # Filter for chromosomes 1-10
    merged_data = merged_data[merged_data['seqid'].str.match(r'^chr([1-9]|10)$')]
    
    # Convert float columns to integers
    float_cols = merged_data.select_dtypes(include=['float64']).columns
    merged_data[float_cols] = merged_data[float_cols].astype(int)
    
    # Create label string from binary columns
    feature_cols = merged_data.columns[3:]
    merged_data['Label'] = [''.join(row) for row in merged_data[feature_cols].astype(str).values]
    merged_data = merged_data[['seqid', 'start', 'end', 'Label']]
    
    # Extract sequences using vectorized operations if possible
    sequences = []
    for _, row in tqdm(merged_data.iterrows(), total=len(merged_data), 
                      desc=f"Extracting sequences (window size: {window_size})"):
        chrom, start, end = row['seqid'], row['start'], row['end']
        sequence = genome.fetch(reference=chrom, start=start, end=end)
        sequences.append(sequence)
    
    merged_data['Seq'] = sequences
    
    merged_data = merged_data.rename(columns={
        'seqid': 'Chr',
        'start': 'Start',
        'end': 'End'
    })[['Chr', 'Start', 'End', 'Seq', 'Label']]

    # remove sequences that are with Ns >= 10%
    cutoff = window_size * 0.1
    merged_data['N_count'] = merged_data['Seq'].apply(lambda seq: seq.upper().count('N'))
    merged_data = merged_data[merged_data['N_count'] < cutoff]
    merged_data = merged_data.drop(columns='N_count')

    # Split into test (chr10) and rest (other chromosomes)
    test_data = merged_data[merged_data['Chr'] == 'chr10'].copy()
    training_data = merged_data[merged_data['Chr'] != 'chr10'].copy()
    
    train_data, valid_data = train_test_split(training_data, test_size = 0.2)
    outputDir = f'../../results/PlantCAD2_tasks/maize_cell_type_accessible_c{window_size}'
    os.makedirs(outputDir, exist_ok=True)
    train_data.to_csv(f'{outputDir}/train.tsv', sep='\t', index=False)
    valid_data.to_csv(f'{outputDir}/valid.tsv', sep='\t', index=False)
    test_data.to_csv(f'{outputDir}/test.tsv', sep='\t', index=False)

Extracting sequences (window size: 1000): 100%|████████████████████████████████████████████████████████| 2068163/2068163 [01:21<00:00, 25502.79it/s]
